# 🧨 Lab 02 — The column had a costume change
Run Lab 01 first. Run this job with its defaults (`feed_table=zops_feed_v1`, `amount_column=amount_cents`). Then switch only `feed_table` to `zops_feed_v2`. The publish cell should fail with `UNRESOLVED_COLUMN`. For recovery, keep v2 and set `amount_column=amount_minor`. Save all three run IDs and the failing task output.

This exercise simulates a breaking upstream rollout by switching input tables. Both revisions stay available for inspection. Let ZeroOps investigate the failed run before pressing Retry. Use the job run history as ground truth if an insight is still pending.


In [ ]:
%sql
SELECT assert_true(:catalog RLIKE '^[A-Za-z_][A-Za-z0-9_]*$' AND :schema RLIKE '^zeroops_lab_[a-z0-9_]+$', 'Use a dedicated zeroops_lab_<alias> schema'),
       assert_true(:feed_table IN ('zops_feed_v1', 'zops_feed_v2'), 'Use only the two lab fixtures'),
       assert_true(:amount_column IN ('amount_cents', 'amount_minor'), 'Use only the two lab amount columns');


In [ ]:
%sql
DESCRIBE TABLE IDENTIFIER(:catalog || '.' || :schema || '.' || :feed_table);


## Review the proposed adapter
The recovery run selects the new amount column explicitly and preserves the published `revenue_cents` name. Expect three USD purchases and 2,500 cents. A rollback to v1 restores service while leaving the new feed unsupported. A production migration that accepts both revisions should normalize each version before aggregation. `COALESCE(amount_cents, amount_minor)` still requires both identifiers to resolve. Verify the currency and units before accepting any mapping.

If a ZeroOps insight appears, record whether it names the incompatible input and cites the failing task or schema. The last cell reconciles the published result against the v1 fixture. Independently inspect the Delta table's History tab to tell a surviving old result from a fresh successful write.


In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :schema || '.zops_daily_revenue') USING DELTA AS
SELECT currency, COUNT(*) AS purchases, SUM(IDENTIFIER(:amount_column)) AS revenue_cents
FROM IDENTIFIER(:catalog || '.' || :schema || '.' || :feed_table)
GROUP BY currency;


In [ ]:
%sql
WITH expected AS (
  SELECT currency, COUNT(*) AS purchases, SUM(amount_cents) AS revenue_cents
  FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_feed_v1') GROUP BY currency
), observed AS (
  SELECT currency, purchases, revenue_cents
  FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_daily_revenue')
)
SELECT COALESCE(o.currency, e.currency) AS currency, o.purchases, o.revenue_cents,
       assert_true(e.currency IS NOT NULL AND o.currency IS NOT NULL
         AND o.purchases = e.purchases AND o.revenue_cents = e.revenue_cents,
         'Published revenue differs from the v1 control') AS reconciliation
FROM expected e FULL OUTER JOIN observed o ON e.currency = o.currency;
